# SIC 2026 · Inteligencia Artificial
## Clase 16: Redes convolucionales y recurrentes a fondo (Capítulo 9, Unidades 1 y 2)

**Samsung Innovation Campus Chile 2026 · ONG Innovacien**

En la Clase 15 entrenamos una CNN y redes recurrentes. Hoy vamos **más adentro**: qué ve realmente una CNN, cómo mejorarla, y qué pasa con una LSTM cuando por fin tiene muchos datos.

| Hora | Parte | Láminas | Qué vamos a hacer |
|---|---|---|---|
| 18:00 a 18:50 | 1. Qué ve una CNN | 5 a 36 | Filtros a mano, stride y padding, y los filtros que aprende una red |
| 18:50 a 20:00 | 2. Mejorar una CNN | 37 a 69 | CIFAR-10: de 70% hacia arriba, cambio por cambio |
| 20:00 a 20:10 | Pausa | | |
| 20:10 a 21:50 | 3. RNN con muchos datos | 70 a 80 | 25.000 reseñas de películas: LSTM, bidireccional y apilada |
| 21:50 a 22:00 | Cierre | | |

**Antes de empezar:** *Entorno de ejecución > Cambiar tipo de entorno de ejecución > GPU (T4)*. Sin GPU, las Partes 2 y 3 tardan mucho.

Hay 12 ejercicios con verificación; las soluciones están al final.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
import keras
from keras import layers
import time, warnings
warnings.filterwarnings("ignore")

def revisar(condicion, mensaje_ok="Correcto.", mensaje_error="Todavía no. Revisa tu respuesta."):
    try:
        print(mensaje_ok if condicion() else mensaje_error)
    except Exception:
        print(mensaje_error)

print("TensorFlow", tf.__version__, "| GPU:", tf.config.list_physical_devices("GPU") or "no")

---
# Parte 1. Qué ve una CNN (láminas 5 a 36)

## Paso 1.1. Un filtro, a mano

Una convolución desliza un **filtro** (una pequeña matriz de pesos) sobre la imagen. En cada posición multiplica el filtro por la zona que cubre, posición por posición, y suma. Si la zona se parece al patrón del filtro, el resultado es alto.

**Qué vamos a hacer:** aplicar a mano un filtro que detecta **bordes verticales** sobre un trozo de 3 × 3.

In [ ]:
zona = np.array([[10, 10, 200],
                 [10, 10, 200],
                 [10, 10, 200]])          # oscuro a la izquierda, claro a la derecha: un borde vertical

filtro_vertical = np.array([[-1, 0, 1],
                            [-1, 0, 1],
                            [-1, 0, 1]])  # resta la izquierda y suma la derecha

producto = zona * filtro_vertical         # multiplicación posición por posición
print("Producto:\n", producto)
print("Suma (el valor del mapa en esta posición):", producto.sum())

**Qué deberías ver:** una suma de **570**: el filtro "se enciende" porque encontró un cambio de oscuro a claro. Sobre una zona pareja (todos 10), daría 0.

### Ejercicio 1
Aplica el mismo `filtro_vertical` a `zona_pareja` (una zona de color uniforme) y guarda el resultado en `valor_pareja`. Luego aplica el filtro de **bordes horizontales** (la transpuesta del vertical, `filtro_vertical.T`) a la `zona` original y guárdalo en `valor_horizontal`.

In [ ]:
zona_pareja = np.full((3, 3), 120)

valor_pareja = None       # TU CÓDIGO AQUÍ
valor_horizontal = None   # TU CÓDIGO AQUÍ
print(valor_pareja, valor_horizontal)

In [ ]:
revisar(lambda: valor_pareja == 0 and valor_horizontal == 0)

**Por qué los dos dan 0:** en una zona pareja no hay bordes; y el borde de `zona` es **vertical**, así que el filtro horizontal no lo detecta. Cada filtro busca un patrón distinto.

## Paso 1.2. Filtros sobre una foto real

**Qué vamos a hacer:** aplicar tres filtros clásicos a una foto completa (incluida en scikit-learn). `tf.nn.conv2d` hace la misma cuenta del Paso 1.1, en todas las posiciones a la vez.

In [ ]:
from sklearn.datasets import load_sample_image

foto = load_sample_image("china.jpg").mean(axis=2).astype("float32") / 255.0     # a escala de grises, de 0 a 1
print("Forma de la foto:", foto.shape)

filtros = {
    "Bordes verticales": np.array([[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]], dtype="float32"),
    "Bordes horizontales": np.array([[-1, -2, -1], [0, 0, 0], [1, 2, 1]], dtype="float32"),
    "Desenfoque": np.full((3, 3), 1 / 9, dtype="float32"),
}

entrada = foto[np.newaxis, :, :, np.newaxis]          # TensorFlow espera (lote, alto, ancho, canales)
fig, ejes = plt.subplots(1, 4, figsize=(16, 3.5))
ejes[0].imshow(foto, cmap="gray"); ejes[0].set_title("Original")
for ax, (nombre, f) in zip(ejes[1:], filtros.items()):
    mapa = tf.nn.conv2d(entrada, f[:, :, np.newaxis, np.newaxis], strides=1, padding="SAME")[0, :, :, 0]
    ax.imshow(np.abs(mapa) if "Bordes" in nombre else mapa, cmap="gray"); ax.set_title(nombre)
for ax in ejes: ax.axis("off")
plt.tight_layout(); plt.show()

**Qué deberías ver:** el filtro vertical resalta los bordes verticales (columnas, árboles), el horizontal los horizontales (techos, el horizonte) y el desenfoque suaviza la foto.

**La idea clave:** estos filtros los diseñó una persona. **Una CNN aprende sus propios filtros**, los que más le sirven para la tarea, y en las capas profundas combina los rasgos simples en patrones complejos. Lo veremos en el Paso 1.4.

## Paso 1.3. Stride y padding (láminas 28 a 36)

| Concepto | Qué es | En Keras |
|---|---|---|
| **Padding** (relleno) | Agregar ceros en el borde para que el filtro quepa en las orillas | `padding="valid"` (sin relleno) o `"same"` (la salida mide lo mismo que la entrada) |
| **Stride** (paso) | Cuántos pixeles avanza el filtro en cada movimiento | `strides=1` (de a uno) o `2` (de a dos: la salida queda a la mitad) |

$$\text{lado de salida} = \left\lfloor \frac{n - k + 2p}{s} \right\rfloor + 1$$

con $n$ el lado de la entrada, $k$ el del filtro, $p$ el relleno y $s$ el paso.

In [ ]:
imagen = tf.zeros((1, 32, 32, 3))
for padding in ["valid", "same"]:
    for stride in [1, 2]:
        salida = layers.Conv2D(8, (3, 3), strides=stride, padding=padding)(imagen)
        print(f"padding={padding:5s} stride={stride} → forma de salida {salida.shape[1:]}")

**Qué deberías ver:** con `valid` y paso 1, 32 pasa a 30; con `same` y paso 1, queda en 32; con paso 2, se reduce a la mitad (15 o 16).

### Ejercicio 2
Con la fórmula: una imagen de **64 × 64**, un filtro de **5 × 5**, sin relleno y con paso **1**. ¿Qué lado tiene la salida? Guárdalo en `lado_a`. ¿Y con el mismo filtro, sin relleno y paso **2**? Guárdalo en `lado_b`.

In [ ]:
lado_a = None   # TU CÓDIGO AQUÍ
lado_b = None   # TU CÓDIGO AQUÍ
print(lado_a, lado_b)

In [ ]:
revisar(lambda: lado_a == 60 and lado_b == 30)

## Paso 1.4. Los filtros que aprende una red

**Qué vamos a hacer:** entrenar rápido una CNN con Fashion-MNIST (la de la Clase 15) y mirar dentro: sus filtros y lo que produce cada uno con una imagen.

In [ ]:
(Xf_tr, yf_tr), (Xf_te, yf_te) = keras.datasets.fashion_mnist.load_data()
Xf_tr = (Xf_tr / 255.0).reshape(-1, 28, 28, 1).astype("float32")
Xf_te = (Xf_te / 255.0).reshape(-1, 28, 28, 1).astype("float32")
PRENDAS = ["Camiseta", "Pantalón", "Suéter", "Vestido", "Abrigo", "Sandalia", "Camisa", "Zapatilla", "Bolso", "Botín"]

keras.utils.set_random_seed(42)
cnn = keras.Sequential([
    layers.Input(shape=(28, 28, 1)),
    layers.Conv2D(16, (3, 3), activation="relu", name="conv1"),
    layers.MaxPooling2D((2, 2)),
    layers.Conv2D(32, (3, 3), activation="relu", name="conv2"),
    layers.MaxPooling2D((2, 2)),
    layers.Flatten(),
    layers.Dense(10, activation="softmax"),
])
cnn.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
cnn.fit(Xf_tr, yf_tr, epochs=2, batch_size=128, verbose=0)
print("Exactitud en prueba:", round(cnn.evaluate(Xf_te, yf_te, verbose=0)[1], 3))

In [ ]:
pesos, sesgos = cnn.get_layer("conv1").get_weights()
print("Forma de los pesos de conv1:", pesos.shape, "→ (alto, ancho, canales de entrada, filtros)")

fig, ejes = plt.subplots(2, 8, figsize=(12, 3.2))
for i, ax in enumerate(ejes.ravel()):
    ax.imshow(pesos[:, :, 0, i], cmap="coolwarm"); ax.set_title(f"filtro {i}", fontsize=8); ax.axis("off")
plt.suptitle("Los 16 filtros de 3 × 3 que aprendió la primera capa"); plt.show()

**Qué deberías ver:** 16 pequeños cuadros de 3 × 3. Varios se parecen a los filtros de bordes del Paso 1.2 (rojo de un lado, azul del otro): **la red descubrió sola que los bordes sirven**.

Ahora, lo que produce cada filtro con una imagen. Para eso armamos un modelo que entrega la salida de la capa `conv1` en vez de la clase:

In [ ]:
espia = keras.Model(inputs=cnn.inputs, outputs=cnn.get_layer("conv1").output)
imagen = Xf_te[0:1]
mapas = espia.predict(imagen, verbose=0)
print("Prenda:", PRENDAS[yf_te[0]], "| Forma de los mapas:", mapas.shape)

fig, ejes = plt.subplots(2, 9, figsize=(14, 3.4))
ejes[0, 0].imshow(imagen[0, :, :, 0], cmap="gray_r"); ejes[0, 0].set_title("original", fontsize=8)
ejes[1, 0].axis("off")
for i, ax in enumerate(np.delete(ejes.ravel(), [0, 9])[:16]):
    ax.imshow(mapas[0, :, :, i], cmap="viridis"); ax.set_title(f"mapa {i}", fontsize=8)
for ax in ejes.ravel(): ax.set_xticks([]); ax.set_yticks([])
plt.show()

**Qué deberías ver:** 16 versiones de la prenda, cada una resaltando algo distinto: el contorno izquierdo, el derecho, la parte superior, las zonas oscuras. Esos 16 mapas son la entrada de la segunda capa, que los combina.

### Ejercicio 3
Arma un modelo `espia2` que entregue la salida de la capa **`conv2`**, aplícalo a `imagen` y guarda en `forma_conv2` la forma del resultado (como tupla). Antes de ejecutarlo, piensa: ¿cuántos mapas tendrá y de qué tamaño?

In [ ]:
forma_conv2 = None  # TU CÓDIGO AQUÍ
print(forma_conv2)

In [ ]:
revisar(lambda: tuple(forma_conv2) == (1, 11, 11, 32))

---
# Parte 2. Mejorar una CNN: CIFAR-10 (láminas 37 a 69)

En la Clase 15, una CNN simple llegó a cerca de 70% con CIFAR-10 (10 clases de objetos a color). Hoy la mejoramos **cambio por cambio**, midiendo cada uno, como se hace en un proyecto real.

In [ ]:
(Xc_tr, yc_tr), (Xc_te, yc_te) = keras.datasets.cifar10.load_data()
yc_tr, yc_te = yc_tr.ravel(), yc_te.ravel()
Xc_tr, Xc_te = Xc_tr.astype("float32") / 255.0, Xc_te.astype("float32") / 255.0
CLASES = ["avión", "auto", "pájaro", "gato", "ciervo", "perro", "rana", "caballo", "barco", "camión"]
print("Entrenamiento:", Xc_tr.shape, "| Prueba:", Xc_te.shape)

EPOCAS = 10          # con GPU, cada modelo tarda 1 a 3 minutos
resultados = {}

def entrenar_y_medir(nombre, modelo, epocas=EPOCAS):
    modelo.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    inicio = time.time()
    h = modelo.fit(Xc_tr, yc_tr, epochs=epocas, batch_size=128, validation_split=0.1, verbose=0)
    exac = modelo.evaluate(Xc_te, yc_te, verbose=0)[1]
    resultados[nombre] = {"exactitud prueba": exac,
                          "exactitud entrenamiento": h.history["accuracy"][-1],
                          "brecha (sobreajuste)": h.history["accuracy"][-1] - h.history["val_accuracy"][-1],
                          "parámetros": modelo.count_params(), "segundos": time.time() - inicio}
    print(f"{nombre}: exactitud en prueba {exac:.3f}")
    return h

**La columna "brecha"** es la diferencia entre la exactitud de entrenamiento y la de validación. Si es grande, el modelo está **memorizando** (sobreajuste).

## Paso 2.1. El punto de partida: la CNN de la Clase 15

In [ ]:
keras.utils.set_random_seed(42)
base = keras.Sequential([
    layers.Input(shape=(32, 32, 3)),
    layers.Conv2D(32, (3, 3), activation="relu", padding="same"), layers.MaxPooling2D(),
    layers.Conv2D(64, (3, 3), activation="relu", padding="same"), layers.MaxPooling2D(),
    layers.Conv2D(64, (3, 3), activation="relu", padding="same"), layers.MaxPooling2D(),
    layers.Flatten(), layers.Dense(64, activation="relu"), layers.Dropout(0.5),
    layers.Dense(10, activation="softmax"),
])
h_base = entrenar_y_medir("1. Base (Clase 15)", base)

## Paso 2.2. Bloques dobles y normalización por lotes

Dos cambios que usan casi todas las CNN modernas:

- **Bloques dobles:** dos convoluciones seguidas antes de cada pooling. La red "mira" dos veces a la misma escala antes de reducir.
- **`BatchNormalization`:** normaliza las salidas de una capa en cada lote (las deja con media cercana a 0 y desviación cercana a 1). Es como el `StandardScaler`, pero **dentro** de la red y en cada capa. Estabiliza y acelera el entrenamiento.

Escribimos una función para no repetir código:

In [ ]:
def bloque(filtros):
    """Dos convoluciones con normalización, y un pooling."""
    return [layers.Conv2D(filtros, (3, 3), padding="same", activation="relu"), layers.BatchNormalization(),
            layers.Conv2D(filtros, (3, 3), padding="same", activation="relu"), layers.BatchNormalization(),
            layers.MaxPooling2D(), layers.Dropout(0.25)]

keras.utils.set_random_seed(42)
profunda = keras.Sequential([layers.Input(shape=(32, 32, 3))] + bloque(32) + bloque(64) + bloque(128) +
                            [layers.Flatten(), layers.Dense(128, activation="relu"), layers.Dropout(0.5),
                             layers.Dense(10, activation="softmax")])
h_prof = entrenar_y_medir("2. Bloques dobles + BatchNorm", profunda)

## Paso 2.3. Aumento de datos

> **Aumento de datos (data augmentation):** crear variaciones de las imágenes de entrenamiento en cada época (voltearlas, rotarlas un poco, desplazarlas). Un gato volteado sigue siendo un gato: la red aprende a reconocerlo aunque cambie de posición, y le cuesta más memorizar.

Keras lo hace con capas que **solo actúan al entrenar** (al predecir, dejan pasar la imagen intacta). Veamos qué hacen:

In [ ]:
aumento = keras.Sequential([
    layers.RandomFlip("horizontal"),            # voltear izquierda-derecha
    layers.RandomRotation(0.05),                # rotar hasta 18 grados
    layers.RandomTranslation(0.1, 0.1),         # desplazar hasta 10%
], name="aumento")

ejemplo = Xc_tr[7:8]
fig, ejes = plt.subplots(1, 8, figsize=(13, 2))
ejes[0].imshow(ejemplo[0]); ejes[0].set_title("original", fontsize=9)
for ax in ejes[1:]:
    ax.imshow(np.clip(aumento(ejemplo, training=True)[0], 0, 1)); ax.set_title("aumentada", fontsize=9)
for ax in ejes: ax.axis("off")
plt.show()

**Qué deberías ver:** la misma imagen 7 veces, cada vez un poco distinta.

### Ejercicio 4
Construye `con_aumento`: la misma red `profunda` del Paso 2.2, pero con la capa `aumento` **justo después del `Input`**. Entrénala con `entrenar_y_medir("3. + Aumento de datos", con_aumento)` y guarda lo que devuelve en `h_aum`.

Pista: `keras.Sequential([layers.Input(...), aumento] + bloque(32) + ...)`.

In [ ]:
keras.utils.set_random_seed(42)
con_aumento = None   # TU CÓDIGO AQUÍ
h_aum = None         # TU CÓDIGO AQUÍ

In [ ]:
revisar(lambda: con_aumento.layers[0].name == "aumento" and "3. + Aumento de datos" in resultados)

## Paso 2.4. Comparar

In [ ]:
tabla = pd.DataFrame(resultados).T
tabla["parámetros"] = tabla["parámetros"].astype(int)
display(tabla.round(3))

plt.figure(figsize=(7, 4))
for nombre, h in [("Base", h_base), ("BatchNorm", h_prof), ("Aumento", h_aum)]:
    if h is not None:
        plt.plot(h.history["val_accuracy"], label=nombre)
plt.xlabel("Época"); plt.ylabel("Exactitud en validación"); plt.legend(); plt.grid(alpha=0.3); plt.show()

**Qué deberías ver** (los números exactos varían):

- La **base** cerca de 0,70, con una brecha que crece: empieza a memorizar.
- Con **bloques dobles y BatchNorm**, un salto claro, cerca de 0,80.
- Con **aumento de datos**, una **brecha mucho menor**. Con solo 10 épocas, su exactitud puede quedar parecida o incluso un poco bajo la anterior, porque aprende más lento; con 30 o más épocas, suele pasar adelante, porque sobreajusta menos.

**La lección:** cada mejora se mide por separado y se mira más de una columna. Una exactitud parecida con mucha menos brecha es un modelo que generalizará mejor.

### Ejercicio 5
Guarda en `mejor_cifar` el nombre del modelo con **mayor exactitud en prueba**, y en `menor_brecha` el nombre del que tiene **menor brecha**. Usa `tabla`.

In [ ]:
mejor_cifar = None   # TU CÓDIGO AQUÍ
menor_brecha = None  # TU CÓDIGO AQUÍ
print(mejor_cifar, "|", menor_brecha)

In [ ]:
revisar(lambda: mejor_cifar == tabla["exactitud prueba"].astype(float).idxmax() and menor_brecha == tabla["brecha (sobreajuste)"].astype(float).idxmin())

### Ejercicio 6
¿Qué le cuesta más al mejor modelo? Con su matriz de confusión, encuentra el par de clases más confundido (sin contar la diagonal) y guárdalo en `par_cifar` como tupla de nombres `(real, predicha)`. Usa el modelo `profunda`.

In [ ]:
from sklearn.metrics import confusion_matrix

par_cifar = None  # TU CÓDIGO AQUÍ
print(par_cifar)

In [ ]:
_cm = confusion_matrix(yc_te, profunda.predict(Xc_te, verbose=0).argmax(axis=1)); np.fill_diagonal(_cm, 0)
_i, _j = np.unravel_index(_cm.argmax(), _cm.shape)
revisar(lambda: tuple(par_cifar) == (CLASES[_i], CLASES[_j]))

**Qué deberías ver:** casi siempre **gato con perro**. Animales del mismo tamaño, en posiciones parecidas, en 32 × 32 pixeles.

---
# Parte 3. RNN con muchos datos: reseñas de películas (láminas 70 a 80)

En la Clase 13, una LSTM entrenada con **60 reseñas** perdió contra una regresión logística con TF-IDF. Dijimos que las redes necesitan muchos datos. Hoy lo comprobamos con **IMDb**: 50.000 reseñas de películas en inglés, positivas o negativas, que vienen con Keras.

## Paso 3.1. Los datos

IMDb ya viene **codificado como números enteros** (Clase 12): cada palabra es su posición en el ranking de frecuencia. Nos quedamos con las 10.000 palabras más comunes.

In [ ]:
VOCAB = 10_000
(Xi_tr, yi_tr), (Xi_te, yi_te) = keras.datasets.imdb.load_data(num_words=VOCAB)
print("Entrenamiento:", len(Xi_tr), "| Prueba:", len(Xi_te), "| Positivas en entrenamiento:", yi_tr.mean())
print("Primera reseña como números:", Xi_tr[0][:15], "...")

indice = keras.datasets.imdb.get_word_index()
palabra_de = {i + 3: p for p, i in indice.items()}          # Keras reserva 0, 1, 2 y 3 para marcas especiales
palabra_de.update({0: "<pad>", 1: "<inicio>", 2: "<desconocida>", 3: "<sin_uso>"})
print("Primera reseña como texto:", " ".join(palabra_de.get(i, "?") for i in Xi_tr[0][:40]), "...")
print("Largo promedio:", int(np.mean([len(r) for r in Xi_tr])), "palabras")

**Qué deberías ver:** 25.000 reseñas para entrenar y 25.000 para probar, mitad positivas y mitad negativas, con unas 240 palabras en promedio.

**Relleno a 200 palabras** (Clase 12): las más largas se cortan y las más cortas se rellenan con ceros.

In [ ]:
LARGO = 200
Si_tr = keras.utils.pad_sequences(Xi_tr, maxlen=LARGO)
Si_te = keras.utils.pad_sequences(Xi_te, maxlen=LARGO)
print("Forma:", Si_tr.shape)

## Paso 3.2. La línea base: bolsa de palabras con regresión logística

Primero el modelo simple, para tener contra qué comparar. Cada reseña pasa a ser un vector de 10.000 posiciones con un 1 en cada palabra que contiene (*multi-hot*).

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

def multi_hot(secuencias, tam=VOCAB):
    M = np.zeros((len(secuencias), tam), dtype="float32")
    for i, s in enumerate(secuencias):
        M[i, s] = 1.0
    return M

inicio = time.time()
logistica = LogisticRegression(max_iter=1000, C=0.1).fit(multi_hot(Xi_tr), yi_tr)
exac_logistica = accuracy_score(yi_te, logistica.predict(multi_hot(Xi_te)))
print(f"Bolsa de palabras + logística: {exac_logistica:.3f} ({time.time() - inicio:.0f} segundos)")
resultados_texto = {"Bolsa de palabras + logística": exac_logistica}

**Qué deberías ver:** cerca de **0,88**. Es una línea base fuerte.

## Paso 3.3. LSTM simple

La arquitectura de la Clase 13, ahora con muchos datos: `Embedding` (cada palabra a un vector de 32 números que se aprende) y una `LSTM` que lee la reseña en orden.

In [ ]:
def red_texto(capas_recurrentes, nombre, epocas=3):
    keras.utils.set_random_seed(42)
    modelo = keras.Sequential([layers.Input(shape=(LARGO,)), layers.Embedding(VOCAB, 32)] + capas_recurrentes +
                              [layers.Dense(1, activation="sigmoid")])
    modelo.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
    inicio = time.time()
    modelo.fit(Si_tr, yi_tr, epochs=epocas, batch_size=128, validation_split=0.2, verbose=0)
    exac = modelo.evaluate(Si_te, yi_te, verbose=0)[1]
    resultados_texto[nombre] = exac
    print(f"{nombre}: {exac:.3f} ({time.time() - inicio:.0f} segundos, {modelo.count_params():,} parámetros)")
    return modelo

lstm_simple = red_texto([layers.LSTM(32)], "LSTM")

**Qué deberías ver:** cerca de **0,85 a 0,88** en 3 épocas. Con 60 reseñas, la LSTM no aprendía; con 25.000, **alcanza a la línea base**.

### Ejercicio 7
¿Cuántos parámetros tiene la capa `Embedding(10000, 32)`? Guárdalo en `params_embedding`. Luego compáralo con el total del modelo: ¿qué capa tiene casi todos los parámetros?

In [ ]:
params_embedding = None  # TU CÓDIGO AQUÍ
print(params_embedding)

In [ ]:
revisar(lambda: params_embedding == 10_000 * 32)

## Paso 3.4. LSTM bidireccional

Una LSTM normal lee de izquierda a derecha. Una **bidireccional** usa **dos** LSTM: una lee hacia adelante y otra hacia atrás, y junta lo que encontró cada una. Así, al decidir sobre una palabra, la red conoce lo que viene antes **y** después. En "the movie was not good at all", la que lee hacia atrás llega a "not" ya sabiendo "good at all".

In [ ]:
lstm_bi = red_texto([layers.Bidirectional(layers.LSTM(32))], "LSTM bidireccional")

### Ejercicio 8
¿Cuántos parámetros tiene `Bidirectional(LSTM(32))` cuando recibe vectores de 32 (la salida del Embedding)? Calcúlalo con la fórmula de la LSTM de la Clase 15 (4 × n × (n + e + 1)) y guárdalo en `params_bi`.

In [ ]:
params_bi = None  # TU CÓDIGO AQUÍ
print(params_bi)

In [ ]:
revisar(lambda: params_bi == 2 * 4 * 32 * (32 + 32 + 1))

## Paso 3.5. LSTM apilada: `return_sequences` (láminas 78 a 80)

Para poner una LSTM sobre otra, la primera debe entregar **una salida por cada palabra** (no solo la memoria final), porque la segunda necesita una secuencia para leer. Eso hace `return_sequences=True`.

| | Salida de `LSTM(32)` | Salida de `LSTM(32, return_sequences=True)` |
|---|---|---|
| Forma | (lote, 32) | (lote, 200, 32) |
| Qué es | La memoria al final de la reseña | La memoria después de cada una de las 200 palabras |

In [ ]:
lstm_apilada = red_texto([layers.LSTM(32, return_sequences=True), layers.LSTM(32)], "LSTM apilada")

### Ejercicio 9
Con la **API funcional** (Clase 15), conecta `keras.Input(shape=(200,))`, `Embedding(10000, 32)` y `LSTM(32, return_sequences=True)`, **sin** más capas, y guarda en `forma_secuencia` la forma de la salida de la LSTM (`salida.shape`, como tupla). Debería coincidir con la tabla.

In [ ]:
forma_secuencia = None  # TU CÓDIGO AQUÍ
print(forma_secuencia)

In [ ]:
revisar(lambda: tuple(forma_secuencia) == (None, 200, 32))

## Paso 3.6. Comparar

In [ ]:
pd.Series(resultados_texto, name="Exactitud en prueba").sort_values(ascending=False).round(3)

**Qué deberías ver:** todos entre 0,85 y 0,89. Las redes **alcanzan** a la línea base, cosa que con 60 reseñas no lograban, pero **no la superan claramente**: la bolsa de palabras ya captura casi todo lo que indica si una reseña es positiva ("great", "awful", "waste").

**Para superar claramente a la línea base** se necesitan modelos que ya aprendieron el idioma con millones de textos y se ajustan a esta tarea: los modelos preentrenados de Hugging Face del Capítulo 10, que llegan a más de 0,93 en IMDb.

### Ejercicio 10
Guarda en `ganador_texto` el nombre del mejor modelo, y en `diferencia_base` la diferencia entre la mejor **red** y la línea base (positiva si la red gana).

In [ ]:
ganador_texto = None     # TU CÓDIGO AQUÍ
diferencia_base = None   # TU CÓDIGO AQUÍ
print(ganador_texto, diferencia_base)

In [ ]:
_redes = {k: v for k, v in resultados_texto.items() if k != "Bolsa de palabras + logística"}
revisar(lambda: ganador_texto == max(resultados_texto, key=resultados_texto.get)
        and np.isclose(diferencia_base, max(_redes.values()) - resultados_texto["Bolsa de palabras + logística"]))

## Paso 3.7. ¿Entiende el orden?

La ventaja teórica de una LSTM es que lee en orden. Probémoslo con frases escritas por nosotros, **en inglés** (el modelo solo conoce el inglés de IMDb).

In [ ]:
def codificar(texto):
    ids = [1] + [indice.get(p, -1) + 3 for p in texto.lower().replace(",", "").replace(".", "").split()]
    ids = [i if 3 < i < VOCAB else 2 for i in ids[1:]]
    return keras.utils.pad_sequences([[1] + ids], maxlen=LARGO)

frases = ["this movie was great", "this movie was not great", "this movie was not bad at all",
          "a waste of time", "i expected a terrible film but it was wonderful"]
tabla_frases = pd.DataFrame({
    "frase": frases,
    "logística (bolsa)": [logistica.predict_proba(multi_hot([codificar(f)[0][codificar(f)[0] > 0]]))[0, 1] for f in frases],
    "LSTM bidireccional": [float(lstm_bi.predict(codificar(f), verbose=0)[0, 0]) for f in frases],
}).round(2)
tabla_frases

Los números son la **probabilidad de que la reseña sea positiva**. Miren "not great" y "not bad at all": la bolsa de palabras ve las mismas palabras sin importar el orden, y la LSTM, en principio, puede usar el orden. Con solo 3 épocas los resultados son irregulares: **a veces la LSTM acierta donde la bolsa falla, y a veces no**. Es un buen punto para discutir: tener la capacidad de leer en orden no garantiza que la red la aproveche.

### Ejercicio 11
Escribe una frase propia en inglés en `mi_frase` y guarda en `prob_mi_frase` la probabilidad de positiva según `lstm_bi`.

In [ ]:
mi_frase = ""          # TU FRASE AQUÍ
prob_mi_frase = None   # TU CÓDIGO AQUÍ
print(mi_frase, prob_mi_frase)

In [ ]:
revisar(lambda: len(mi_frase.split()) >= 3 and np.isclose(prob_mi_frase, float(lstm_bi.predict(codificar(mi_frase), verbose=0)[0, 0]), atol=1e-4))

### Ejercicio 12
¿Qué modelo es **más rápido de entrenar** y cuál **más fácil de explicar**? Escribe tu respuesta en la celda de texto siguiente. (No tiene verificación: discútanlo en grupo.)

**Mi respuesta:** (doble clic para escribir)

---
# Cierre

| Parte | Lo esencial |
|---|---|
| Filtros | Una convolución es multiplicar y sumar. La CNN aprende sus filtros, y los primeros parecen detectores de bordes. |
| Stride y padding | Controlan el tamaño de salida: (n − k + 2p) / s + 1. |
| Mejorar una CNN | Bloques dobles y BatchNorm suben la exactitud; el aumento de datos reduce el sobreajuste. Medir cada cambio. |
| RNN con muchos datos | Con 25.000 reseñas, la LSTM alcanza a la bolsa de palabras. Para superarla claramente se necesitan modelos preentrenados (Capítulo 10). |
| Bidireccional y apilada | Leer en los dos sentidos; `return_sequences=True` para poner una LSTM sobre otra. |

Guarden el notebook en `ejercicios_estudiantes` como `clase16_su_nombre.ipynb`.

---
# Soluciones

### Ejercicio 1
En la zona pareja, cada fila suma −120 + 0 + 120 = 0. Con el filtro horizontal, las tres filas de `zona` son iguales: la fila de arriba resta lo mismo que suma la de abajo.

In [ ]:
zona_pareja = np.full((3, 3), 120)
valor_pareja = (zona_pareja * filtro_vertical).sum()
valor_horizontal = (zona * filtro_vertical.T).sum()
print(valor_pareja, valor_horizontal)

### Ejercicio 2
(64 − 5 + 0) / 1 + 1 = 60. Con paso 2: (64 − 5) / 2 = 29,5, que se redondea hacia abajo a 29, y más 1 = 30.

In [ ]:
lado_a = (64 - 5 + 2 * 0) // 1 + 1
lado_b = (64 - 5 + 2 * 0) // 2 + 1
print(lado_a, lado_b)

### Ejercicio 3
conv1 deja 26 × 26 con 16 mapas; el pooling, 13 × 13; conv2 (3 × 3 sin relleno), 11 × 11 con 32 mapas.

In [ ]:
espia2 = keras.Model(inputs=cnn.inputs, outputs=cnn.get_layer("conv2").output)
forma_conv2 = espia2.predict(imagen, verbose=0).shape
print(forma_conv2)

### Ejercicio 4

In [ ]:
keras.utils.set_random_seed(42)
con_aumento = keras.Sequential([layers.Input(shape=(32, 32, 3)), aumento] + bloque(32) + bloque(64) + bloque(128) +
                               [layers.Flatten(), layers.Dense(128, activation="relu"), layers.Dropout(0.5),
                                layers.Dense(10, activation="softmax")])
h_aum = entrenar_y_medir("3. + Aumento de datos", con_aumento)

### Ejercicio 5
`idxmax` e `idxmin` devuelven el nombre de la fila (el índice), no el valor. Se convierte a `float` porque la tabla puede quedar con tipo texto.

In [ ]:
mejor_cifar = tabla["exactitud prueba"].astype(float).idxmax()
menor_brecha = tabla["brecha (sobreajuste)"].astype(float).idxmin()
print(mejor_cifar, "|", menor_brecha)

### Ejercicio 6

In [ ]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(yc_te, profunda.predict(Xc_te, verbose=0).argmax(axis=1))
np.fill_diagonal(cm, 0)
i, j = np.unravel_index(cm.argmax(), cm.shape)
par_cifar = (CLASES[i], CLASES[j])
print(par_cifar, cm[i, j])

### Ejercicio 7
Una fila de 32 números por cada una de las 10.000 palabras: 320.000. Es casi todo el modelo: la LSTM tiene 8.320 y la salida 33.

In [ ]:
params_embedding = 10_000 * 32
print(params_embedding, "| total del modelo:", lstm_simple.count_params())

### Ejercicio 8
Cada LSTM: 4 × 32 × (32 + 32 + 1) = 8.320. La bidireccional tiene dos: 16.640.

In [ ]:
params_bi = 2 * 4 * 32 * (32 + 32 + 1)
print(params_bi)

### Ejercicio 9

In [ ]:
entrada = keras.Input(shape=(200,))
x = layers.Embedding(10_000, 32)(entrada)
salida = layers.LSTM(32, return_sequences=True)(x)
forma_secuencia = tuple(salida.shape)
print(forma_secuencia)

### Ejercicio 10

In [ ]:
ganador_texto = max(resultados_texto, key=resultados_texto.get)
redes = {k: v for k, v in resultados_texto.items() if k != "Bolsa de palabras + logística"}
diferencia_base = max(redes.values()) - resultados_texto["Bolsa de palabras + logística"]
print(ganador_texto, round(diferencia_base, 3))

### Ejercicio 11
Cualquier frase de 3 o más palabras en inglés.

In [ ]:
mi_frase = "the actors were good but the story was boring"
prob_mi_frase = float(lstm_bi.predict(codificar(mi_frase), verbose=0)[0, 0])
print(round(prob_mi_frase, 3))

### Ejercicio 12
La logística entrena en segundos y se explica mirando sus coeficientes (qué palabras empujan a positiva o negativa). Las LSTM tardan minutos con GPU, y sus 330.000 parámetros no se pueden leer uno por uno. Con exactitudes parecidas, en un proyecto real conviene la logística, salvo que el orden de las palabras sea clave.